# Train MACEField on BaTiO₃ reference data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/02_train_macefield_batio3.ipynb)


This notebook trains and evaluates a field-conditioned MACE model using the copied BaTiO₃ reference-calculation data in this course’s `data/` folder. The labels are stored calculation results copied from the source dataset.

The data contain 59 periodic 135-atom configurations. The seed-23 split has 48 training and 11 validation configurations. The preprocessing notebook in the source repository does not preserve the electronic-structure code or functional, so this tutorial does not assign an XC method that is absent from the local record.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def clone_checkout(path, url, branch):
    path = Path(path)
    if not (path / ".git").is_dir():
        if path.exists() and any(path.iterdir()):
            raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", branch, url, str(path)],
            check=True,
        )


if IN_COLAB:
    REPO = Path("/content/mace-field")
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    clone_checkout(REPO, PACKAGE_URL, "clean")
    clone_checkout(TUTORIAL_REPO, TUTORIAL_URL, "main")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--editable", str(REPO)],
        check=True,
    )
else:
    CWD = Path.cwd().resolve()
    candidates = (CWD, *CWD.parents)
    TUTORIAL_REPO = next(
        (path for path in candidates
         if (path / "tutorials/scripts/bootstrap.py").is_file()),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run from a mace-field-tutorial checkout, or open this notebook in Colab."
        )
    package_candidates = [
        Path(os.environ["MACEFIELD_REPO"]).expanduser().resolve()
        if os.environ.get("MACEFIELD_REPO") else None,
        TUTORIAL_REPO.parent / "mace-field",
        *candidates,
    ]
    REPO = next(
        (path.resolve() for path in package_candidates
         if path is not None and (path / "mace/modules/extensions.py").is_file()),
        None,
    )
    if REPO is None:
        raise RuntimeError(
            "Clone mdi-group/mace-field beside this tutorial checkout, or set MACEFIELD_REPO."
        )
    os.environ["MACEFIELD_REPO"] = str(REPO)

os.environ["MACEFIELD_REPO"] = str(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(TUTORIAL_REPO))
print("MACEField source:", REPO)
print("Tutorial assets:", TUTORIAL_REPO)
print("Runtime:", "Google Colab" if IN_COLAB else "local Jupyter")


## Learning goals

- inspect the copied ExtXYZ labels and their provenance;
- train `--model=MACEField` with energy, force, field, and polarization supervision;
- evaluate held-out errors in physical units; and
- identify which uses the training data can and cannot support.

Start Jupyter inside this checkout or one of its subdirectories. Install this checkout and its base dependencies in the active kernel before training.

In [ ]:
import json
import os
import sys
import subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA = TUTORIAL_DIR / "data"
train = read(DATA / "macefield_batio3_reference_train.extxyz", index=":")
valid = read(DATA / "macefield_batio3_reference_valid.extxyz", index=":")
manifest = json.loads((DATA / "macefield_batio3_reference_split.json").read_text())
assert len(train) == 48 and len(valid) == 11
assert all(len(a) == 135 for a in train + valid)
print(f"train frames: {len(train)} | validation frames: {len(valid)}")
print("source:", manifest["source_file"])
print("source SHA-256:", manifest["source_sha256"])
print("info labels:", sorted(train[0].info))
print("array labels:", sorted(train[0].arrays))


## Dataset provenance and coverage

The source is `MACEField_dev/BaTiO3/BaTiO3-preprocessed.xyz`, prepared from stored ASE calculator results by `MACEField_dev/BaTiO3/process-BaTiO3-data.ipynb`. The copied manifest records the source checksum, the original seed-23 validation indices, and the derived train/validation file checksums. It is a calculated reference dataset, not an experimental dataset.

Every frame has the same field tag, `[0.0036, 0.0036, 0.0036] V/Å`. This gives the model a real field-conditioned label, but it does not sample field reversal, a range of field magnitudes, or a switching path. Validation therefore measures held-out structures at the same field. It is not a field-transfer or hysteresis validation. There is no independent test split.

In [ ]:
all_frames = train + valid
field_vectors = np.asarray([a.info["REF_electric_field"] for a in all_frames], dtype=float)
unique_fields = np.unique(np.round(field_vectors, decimals=10), axis=0)
field_direction = field_vectors[0] / np.linalg.norm(field_vectors[0])
train_energy_pa = np.asarray([a.info["REF_energy"] / len(a) for a in train])
valid_energy_pa = np.asarray([a.info["REF_energy"] / len(a) for a in valid])
train_p_parallel = np.asarray([
    np.dot(a.info["REF_polarization"], field_direction) for a in train
])
valid_p_parallel = np.asarray([
    np.dot(a.info["REF_polarization"], field_direction) for a in valid
])
print("unique training field vectors (V/Å):\n", unique_fields)
print("cell formula:", train[0].get_chemical_formula(), "| atoms:", len(train[0]))
print("cell volume (Å³):", train[0].get_volume())
print("energy range (eV/atom):", float(train_energy_pa.min()), "to", float(train_energy_pa.max()))
print("field-parallel polarization range (e/Å²):",
      float(min(train_p_parallel.min(), valid_p_parallel.min())), "to",
      float(max(train_p_parallel.max(), valid_p_parallel.max())))


## Label-to-trainer mapping

| ExtXYZ label | Trainer option | Units | Meaning |
| --- | --- | --- | --- |
| `REF_energy` | `--energy_key` | eV per cell | Reference energy |
| `REF_forces` | `--forces_key` | eV Å⁻¹ | Reference forces |
| `REF_electric_field` | `--electric_field_key` | V Å⁻¹ | Field associated with each frame |
| `REF_polarization` | `--polarization_key` | e Å⁻² | Field derivative of energy, divided by cell volume |
| `REF_becs` | `--becs_key` | e | Born effective charge tensors; optional higher-order target |
| `REF_polarizability` | `--polarizability_key` | dimensionless | Electric susceptibility target; optional higher-order target |

The `REF_stress` labels are retained in the source, but this notebook does not train on stress. The default training run uses energy, forces, and polarization. BEC and susceptibility training can be enabled with `MACEFIELD_TUTORIAL_FULL_RESPONSES=1`; these response derivatives cost more, especially for 135-atom cells.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
energy_offset = np.mean(train_energy_pa)
ax.scatter((train_energy_pa - energy_offset) * 1000, train_p_parallel,
           s=26, label="train")
ax.scatter((valid_energy_pa - energy_offset) * 1000, valid_p_parallel,
           s=36, marker="s", label="validation")
ax.set(xlabel="energy per atom, centered on training mean (meV/atom)",
       ylabel="polarization along [111] field (e/Å²)",
       title="Structural coverage at one field vector")
ax.grid(alpha=0.2)
ax.legend()
plt.show()
print("This plot shows structural variation only; all points share one field vector.")


## Train from scratch

The next cell uses the checked-in train and validation files, so the training script cannot create a new random split. The default is 100 epochs, one 135-atom structure per batch, and force weight 100. Set `MACEFIELD_TUTORIAL_EPOCHS` and `MACEFIELD_TRAIN_DEVICE` to adjust the run. The model is saved in a separate run directory with `--save_cpu` for portable loading.

The small dataset and limited field coverage are deliberate teaching constraints. A successful optimization or low validation error does not establish switching accuracy.

In [ ]:
import torch

RUN_DIR = TUTORIAL_DIR / "runs/bto_macefield_training"
RUN_DIR.mkdir(parents=True, exist_ok=True)
DEFAULT_EPOCHS = "10" if IN_COLAB else "100"
EPOCHS = int(os.environ.get("MACEFIELD_TUTORIAL_EPOCHS", DEFAULT_EPOCHS))
DEVICE = os.environ.get("MACEFIELD_TRAIN_DEVICE",
                        "cuda" if torch.cuda.is_available() else "cpu")
env = os.environ.copy()
full_responses = os.environ.get("MACEFIELD_TUTORIAL_FULL_RESPONSES", "0") == "1"
env.update({
    "PYTHON": sys.executable,
    "DEVICE": DEVICE,
    "EPOCHS": str(EPOCHS),
    "BATCH_SIZE": env.get("MACEFIELD_TUTORIAL_BATCH_SIZE", "1"),
    "FORCES_WEIGHT": env.get("MACEFIELD_TUTORIAL_FORCES_WEIGHT", "100"),
    "COMPUTE_BECS": str(full_responses),
    "COMPUTE_POLARIZABILITY": str(full_responses),
    "BECS_WEIGHT": "10" if full_responses else "0",
    "POLARIZABILITY_WEIGHT": "1" if full_responses else "0",
    "OUT_DIR": str(RUN_DIR),
})
print("Training device:", DEVICE, "| epochs:", EPOCHS,
      "| batch size:", env["BATCH_SIZE"],
      "| full response targets:", full_responses, "| output:", RUN_DIR)
if IN_COLAB and EPOCHS < 100:
    print("Short Colab demonstration fit; set MACEFIELD_TUTORIAL_EPOCHS=100 for the longer tutorial run.")
subprocess.run(["bash", str(TUTORIAL_DIR / "scripts/train_macefield.sh")],
               cwd=TUTORIAL_REPO, env=env, check=True)
MODEL = RUN_DIR / "MACEField-BaTiO3-reference_run-23.model"
assert MODEL.is_file(), f"Training completed without the expected checkpoint: {MODEL}"
print("checkpoint:", MODEL)


## Evaluate the held-out structures

Report each property separately. Energy errors are normalized per atom because every cell has the same 135 atoms. The validation structures share the training field vector, so these metrics do not measure response to field reversal or extrapolation.

In [ ]:
from mace.calculators import MACECalculator

pred_e, ref_e = [], []
pred_f, ref_f = [], []
pred_p, ref_p = [], []
for atoms_ref in valid:
    atoms = atoms_ref.copy()
    atoms.calc = MACECalculator(
        model_paths=str(MODEL), model_type="MACEField", device=DEVICE,
        default_dtype="float64",
        electric_field=np.asarray(atoms_ref.info["REF_electric_field"], dtype=float),
        compute_polarization=True, compute_becs=False,
        compute_polarizability=False,
    )
    pred_e.append(atoms.get_potential_energy())
    ref_e.append(atoms_ref.info["REF_energy"])
    pred_f.append(atoms.get_forces())
    ref_f.append(atoms_ref.arrays["REF_forces"])
    pred_p.append(atoms.calc.results["polarization"])
    ref_p.append(atoms_ref.info["REF_polarization"])

pred_e, ref_e = np.asarray(pred_e), np.asarray(ref_e)
pred_f, ref_f = np.asarray(pred_f), np.asarray(ref_f)
pred_p, ref_p = np.asarray(pred_p), np.asarray(ref_p)
n_atoms = np.asarray([len(a) for a in valid], dtype=float)
energy_error_mev_atom = (pred_e - ref_e) / n_atoms * 1000
rmse_energy_mev_atom = np.sqrt(np.mean(energy_error_mev_atom**2))
rmse_force_mev_A = np.sqrt(np.mean((pred_f - ref_f)**2)) * 1000
rmse_pol_e_A2 = np.sqrt(np.mean((pred_p - ref_p)**2))
print(f"validation energy RMSE: {rmse_energy_mev_atom:.3f} meV/atom")
print(f"validation force RMSE: {rmse_force_mev_A:.3f} meV/Å")
print(f"validation polarization RMSE: {rmse_pol_e_A2:.6g} e/Å²")


In [ ]:
ref_e_pa = ref_e / n_atoms
pred_e_pa = pred_e / n_atoms
energy_center = np.mean(ref_e_pa)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].scatter((ref_e_pa - energy_center) * 1000,
                (pred_e_pa - energy_center) * 1000, s=24)
axes[0].set(xlabel="reference (centered meV/atom)",
            ylabel="model (centered meV/atom)", title="Energy")
axes[1].scatter(ref_f.ravel(), pred_f.ravel(), s=10)
axes[1].set(xlabel="reference (eV/Å)", ylabel="model (eV/Å)", title="Force")
axes[2].scatter(ref_p.ravel(), pred_p.ravel(), s=24)
axes[2].set(xlabel="reference (e/Å²)", ylabel="model (e/Å²)", title="Polarization")
for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## How to interpret the fit

These are held-out structural errors from an 11-frame validation split at one field. They can check whether the training route learned these labels, but they cannot test a ferroelectric double well, negative-field behavior, a switching path, or thermal stability. Do not infer a coercive field or a physical hysteresis loop from this dataset.

For predictive field-driven MD, add reference calculations across both field polarities and magnitudes, thermal distortions, and switching configurations; retain an independent test split and complete electronic-structure and polarization-branch provenance.

## Next

- [MACEField architecture and response derivatives](01_macefield_architecture.ipynb) uses this trained checkpoint for a finite-difference derivative check.
- [Finite-field MLMD field-cycle workflow](03_finite_field_mlmd_hysteresis.ipynb) initializes from a real 135-atom structure and labels its limited-field cycle as exploratory, because the available labels do not validate reversal or switching.